In [16]:
with open("../gcode/cylinder.gcode", "r") as file:
    gcode_lines = file.readlines()

In [17]:
accepted_codes = ("G0", "G1", "G90", "G91")

filtered_lines = list(filter(lambda l: l.startswith(accepted_codes), gcode_lines))

In [18]:
rejected_actuations = ("E", "F")

def clean_gcode_line(l):
    split_actuations = l.split(" ")
    cleaned_actuations = list(filter(lambda l: not l.startswith(rejected_actuations), split_actuations))
    joined_actuations = " ".join(cleaned_actuations)
    return joined_actuations

parsed_lines = list(map(lambda l: clean_gcode_line(l), filtered_lines))

In [19]:
from geometry_msgs.msg import Pose

accepted_codes = ("G0", "G1", "G90", "G91")
rejected_actuations = ("E", "F")

def clean_gcode_line(l):
    split_actuations = l.split(" ")
    cleaned_actuations = list(filter(lambda l: not l.startswith(rejected_actuations), split_actuations))
    joined_actuations = " ".join(cleaned_actuations)
    return joined_actuations


def generate_waypoints(gcode_lines: list[str]):
    # Holds the presently parsed waypoint
    coords = {"x": 0.0, "y": 0.0, "z": 0.0}
    waypoints = []

    for line in gcode_lines:
        for code in line.split(" "):
            match code[0]:
                case "X":
                    coords["x"] = float(code[1:])
                case "Y":
                    coords["y"] = float(code[1:])
                case "Z":
                    coords["z"] = float(code[1:])

        pose = Pose()
        pose.position.x = coords["x"]
        pose.position.y = coords["y"]
        pose.position.z = coords["z"]

        pose.orientation.x = 0.0
        pose.orientation.y = 1.0
        pose.orientation.z = 0.0
        pose.orientation.w = 0.0

        waypoints.append(pose)

    return waypoints


def gcode_to_waypoints(path):
    with open("../assets/cylinder.gcode", "r") as file:
        gcode_lines = file.readlines()

    filtered_lines = list(filter(lambda l: l.startswith(accepted_codes), gcode_lines))
    parsed_lines = list(map(lambda l: clean_gcode_line(l), filtered_lines))
    waypoints = generate_waypoints(parsed_lines)
    return waypoints

In [20]:

waypoints = generate_waypoints(parsed_lines)

In [30]:
import numpy as np
import plotly.express as px

sample_waypoints = waypoints[14:-2]

x = np.array([w.position.x for w in sample_waypoints])
y = np.array([w.position.y for w in sample_waypoints])
z = np.array([w.position.z for w in sample_waypoints])
s = np.arange(len(sample_waypoints))

fig = px.scatter_3d(
    x=x,
    y=y,
    z=z,
    color=s,
    color_continuous_scale="Magma",
    labels={
        "x": "X Axis (mm)",
        "y": "Y Axis (mm)", 
        "z": "Z Axis (mm)",
        "color": "Sequential order"},
    title="Interactive Plotly 3D Scatter Plot with 4D Intensity",
    opacity=0.8,
)

fig.update_layout(
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.update_traces(marker=dict(size=5))

fig.show()
